In [2]:
import pygame as pg
import numpy as np
import random
pg.init()

#Grid creation

#Grid & Cell dimensions
grid_size = 65
cell_size = 15
width = height = grid_size * cell_size

#colors
white = (255,255,255)
black = (0,0,0)
gray = (200,200,200)
red = (255,0,0)

#Creating Simulation - Window
window = pg.display.set_mode((width,height))
pg.display.set_caption("Langton's Ant")

#Grid
grid = np.zeros((grid_size, grid_size), dtype=int)  
def draw_grid():
    for row in range(grid_size):
        for col in range(grid_size):
            rect = pg.Rect(col * cell_size, row * cell_size, cell_size, cell_size)

            #Flipping of color of the block
            if grid[row][col] == 1 :
                color = black
            else :
                color = white
            #Pheromone (or path) of Ants
            if (col,row) in phero :
                if "A" in phero[(col,row)] or "B" in phero[(col,row)]:
                    color = gray #Pheromone color
                elif "A" in phero[(col,row)] and "B" in phero[(col,row)]:
                    color = black #Both on block
            
            pg.draw.rect(window, color, rect)
            pg.draw.rect(window, gray, rect, 1)

#Pheromone info in dictionary
phero = {} #Key - (x,y) : Value - {"A" :<Pheromone level>,"B" : <Pheromone level>}

#Defining Ant & it's movement
class Ant:
    def __init__(self,x,y,direct,name):
        self.x = x
        self.y = y
        self.direct = direct #0 - Up;1 - Right;2 - Down;3 - Left
        self.name = name
        
    def move(self,grid):
        straight = False #Default movement of Ant
        
        #Checking pheromone at current position and deciding Ant behaviour
        if (self.x,self.y) in phero and self.name in phero[(self.x,self.y)]: #Self pheromone detected
            if random.random() < 0.8 : #80% probability to move straight
                straight = True #Ant moves straight 80% of the time
                
        elif (self.x,self.y) in phero: #Other's pheromone detected
            if random.random() < 0.2: #20% probability to move straight
                straight = True #Ant moves straight 20% of the time
        #Normal movement if Ant is not moving straight
        if not straight :
            if grid[self.y][self.x] == 0: #0 - white
                grid[self.y][self.x] = 1  #1 - updating it to black
                self.direct = (self.direct + 1) % 4 #Turns 90deg clockwise
            else :
                grid[self.y][self.x] = 0
                self.direct = (self.direct - 1) % 4 #Turns 90deg counter - clockwise
                
        
        
        #Storing pheromone at current position
        if (self.x,self.y) not in phero:
            phero[(self.x,self.y)] = {self.name : 0}
        phero[(self.x, self.y)][self.name] = 5  #Pheromone lasts 5 steps

        #Ant moves forward
        if self.direct == 0: 
            self.y -= 1
        elif self.direct == 1:
            self.x += 1
        elif self.direct == 2:
            self.y += 1
        elif self.direct == 3:
            self.x -= 1

        #Ensuring that Ant does not goes outside the grid
        self.x %= len(grid[0])  #horizontally
        self.y %= len(grid)     #vertically

    #Pheromone decay over-time
    @staticmethod
    def decay():
        remove = []
        for pos in list(phero.keys()): 
            for ant in list(phero[pos]): #Iterating over position(x,y) in list format
                phero[pos][ant] -= 1
                if phero[pos][ant] == 0:
                    del phero[pos][ant] #Removed pheromone if it reaches zero
            if not phero[pos]:
                remove.append(pos) #Store for deletion
        for pos in remove:
            del phero[pos] #Pheromone clean-up for the position

        
    

A = Ant(grid_size // 2, grid_size // 2, 0,"A") #Initialized the Ant A
B = Ant(grid_size // 3,grid_size // 3, 1,"B") #Initialized the Ant B


#Simulation specific variable
running = True

#Simulation loop
while running :
    window.fill(black)
    draw_grid()

    A_rect = pg.Rect(A.x * cell_size, A.y * cell_size, cell_size, cell_size)
    pg.draw.rect(window, red,A_rect) #Drawing the Ant A
    B_rect = pg.Rect(B.x * cell_size, B.y * cell_size, cell_size, cell_size)
    pg.draw.rect(window, red,B_rect) #Drawing the Ant B

    
    for event in pg.event.get():
        if event.type == pg.QUIT:
            running = False

    A.move(grid) #Movement
    B.move(grid)
    Ant.decay() #For decay of pheromone
    
    pg.display.flip() #Updating display
    #pg.time.delay(100) #Slowed down

pg.quit()
quit()

